<a href="https://colab.research.google.com/github/Kimballa05/agentic-ai-jetstream2-bioinformatics/blob/main/agentic_ai_ecoli_wgs_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic AI Workshop — *E. coli* WGS Quality Control
### Hands-on participant notebook | Jetstream2 JupyterHub + OpenCode

**Your mission:** You are part of a research team checking whether several *E. coli* sequencing datasets are suitable for further analysis. With an AI coding agent, you will inspect the sample records, assess sequencing quality, clean the reads, explore the results, and produce a short report. One of the four accessions is an intentional mismatch.

**How to use this notebook:** Work from top to bottom. At each step, read the task, give the **Ask your agent** prompt to OpenCode in the JupyterLab terminal, examine its work, and answer the **Explore / Checkpoint** question. You do **not** need to write Python from scratch. The code cells below provide a *reference implementation* or inspection tools—not an automatic test of the agent, and not a requirement to copy code into the agent. You can use the reference code if your agent needs help.

**Workshop setup:** The organizers plan to provide a JupyterHub workspace with tools already installed, shared read-only input data, and your own writable project folder. No live NCBI download is required for the main exercise. The exact shared-data path and OpenCode authentication will be confirmed by the instructors. Bioinformatics and agentic-AI background are covered in the accompanying slides.

**Safety rule:** Ask the agent to show its plan before executing commands; do not delete raw data, change shared files, download large datasets, or install software without instructor approval.

## What you will do

```text
Four accessions (maybe more) + provided metadata       (Step 1: identify valid WGS)
                  |
                  v
       Select WGS samples only             (Step 2: locate staged reads)
                  |
                  v
        Inspect raw FASTQ files            (Step 3: raw QC)
                  |
                  v
           Clean with fastp                (Step 4: preprocessing)
                  |
                  v
        Compare before vs after            (Step 5: QC + visualization)
                  |
                  v
        Extract QC measurements            (Step 6: feature table)
                  |
                  v
     Small illustrative ML classifier      (Step 7: PASS / REVIEW)
                  |
                  v
         Interpret and report              (Step 8: final summary)
```

**Success looks like:** three (maybe more) eligible WGS samples (subject to metadata verification), one flagged mismatch, readable QC summaries and plots, a completed feature table, and a short report explaining the results.

## Workshop accessions

The expected project is paired-end *E. coli* WGS.

| sample_id | run_accession |
|---|---|
| sample01 | `ERR4567643` |
| sample02 | `SRR25073407` |
| sample03 | `SRR29154704` |
| sample04 | `SRR27836258` |

The participant-facing workflow should not identify the mismatch in advance.

In [ ]:
import pandas as pd
from pathlib import Path

samples = pd.DataFrame({
    "sample_id": ["sample01", "sample02", "sample03", "sample04"],
    "run_accession": [
        "ERR4567643",
        "SRR25073407",
        "SRR29154704",
        "SRR27836258",
    ],
})

BASE = Path("agentic_ecoli_wgs")

for d in [
    "metadata/xml",
    "raw",
    "processed",
    "qc/raw/fastqc",
    "qc/raw/multiqc",
    "qc/processed/fastqc",
    "qc/processed/multiqc",
    "reports/fastp",
    "logs",
]:
    (BASE / d).mkdir(parents=True, exist_ok=True)

for sid in samples["sample_id"]:
    (BASE / "raw" / sid).mkdir(parents=True, exist_ok=True)
    (BASE / "processed" / sid).mkdir(parents=True, exist_ok=True)

samples.to_csv(BASE / "samples.tsv", sep="\t", index=False)
samples

,sample_id,run_accession
0,sample01,ERR4567643
1,sample02,SRR25073407
2,sample03,SRR29154704
3,sample04,SRR27836258


## Project files and responsibilities

```text
agentic_ecoli_wgs/
├── samples.tsv                    # accession list
├── metadata/                      # staged metadata, review, selected WGS
├── raw/<sample_id>/               # input FASTQ (may be links to shared files)
├── processed/<sample_id>/         # cleaned FASTQ
├── qc/raw/ and qc/processed/      # SeqKit, FastQC, MultiQC outputs
├── reports/                       # plots, feature table, predictions, report
└── logs/                          # commands and versions
```

`metadata/` contains input information, `raw/` untouched reads, `processed/` cleaned reads, `qc/` QC outputs, `reports/` summary tables and ML results, and `logs/` executed commands.

**File cheat sheet:** `.tsv` = spreadsheet-like tab-separated text; `.fastq.gz` = compressed sequencing reads with quality scores; `.html` = browser-viewable QC report; `.json` = structured tool output; `.md` = Markdown report.

**Who does what?** You choose questions and evaluate outputs. OpenCode proposes and runs work, with your approval. The code cells in this notebook are known examples or lightweight output viewers. They are not the expected agent-generated answer.


## Workspace setup and optional Colab testing

**Jetstream2 workshop:** The tools should already be inside each participant's JupyterHub container. The next installation cell is **only for optional Colab testing**; skip it on Jetstream2. Open a JupyterLab terminal, launch `opencode` using instructor instructions, and keep this notebook open beside it. The API key should be provided through a secure, instructor-approved mechanism, not pasted into the notebook.

For Colab, the prepared FASTQ files and metadata must still be made available separately. Live retrieval is intentionally not part of the core exercise.

In [ ]:
# Install a small tool set for the Colab prototype
!apt-get install -y -qq sra-toolkit fastqc fastp pigz > /dev/null
!pip install -q multiqc pandas scikit-learn pyyaml

# Step 1 — Validate the sequencing accessions

Before analyzing sequencing data, we need to verify that each accession corresponds to the expected organism and sequencing experiment. The agent will retrieve metadata from NCBI SRA and examine information such as organism, library strategy, and sequencing layout. Samples that do not meet our paired-end E. coli WGS requirements will be flagged for exclusion or manual review. This prevents unsuitable datasets from entering downstream analyses.

**Goal:** Determine which datasets belong in a paired-end *E. coli* WGS experiment. One accession belongs to a different sequencing approach, so do not assume all four are appropriate. The instructor provides SRA metadata to avoid live network access.

### Ask your agent

> Read `agentic_ecoli_wgs/samples.tsv` and the provided `metadata/sra_metadata.tsv`. Review organism, library strategy, source, layout, and experiment description. Write `metadata/metadata_review.tsv` with `sample_id`, `run_accession`, `status` (`VALID_WGS`, `MISMATCH`, or `REVIEW`), and brief evidence. Show the table and explain uncertain cases. Do not query NCBI or process reads yet.

**Explore / Checkpoint:** Which field was most helpful in distinguishing WGS from the mismatched run? Does the explanation agree with the evidence? **Deliverable:** `metadata/metadata_review.tsv`.

*Reference below:* metadata retrieval and XML parsing are optional fallback examples, not required for the staged-data exercise.

### Optional reference: live NCBI XML retrieval (not part of the main workshop)

These cells show how an extended workflow could retrieve metadata using Entrez Direct. They are disabled by default. **For the live workshop, use the staged `sra_metadata.tsv`.**


In [ ]:
import subprocess

RUN_METADATA_RETRIEVAL = False  # Optional instructor demo only; leave False for workshop.

if RUN_METADATA_RETRIEVAL:
    for _, row in samples.iterrows():
        sample = row["sample_id"]
        accession = row["run_accession"]
        out_xml = BASE / "metadata" / "xml" / f"{sample}_{accession}.xml"

        cmd = (
            f'esearch -db sra -query "{accession}" '
            f'| efetch -format native -mode xml '
            f'> "{out_xml}"'
        )

        subprocess.run(cmd, shell=True, check=True)

        with open(BASE / "logs" / "commands.log", "a") as log:
            log.write(cmd + "\n")

        print(f"Saved {out_xml}")
else:
    print("Optional XML retrieval disabled; use staged metadata for workshop.")

In [ ]:
import xml.etree.ElementTree as ET

def get_text(root, path):
    el = root.find(path)
    return "" if el is None or el.text is None else el.text.strip()

def parse_sra_xml(sample, accession):
    xml_file = BASE / "metadata" / "xml" / f"{sample}_{accession}.xml"
    root = ET.parse(xml_file).getroot()

    layout_node = root.find(".//LIBRARY_LAYOUT")
    layout = ""
    if layout_node is not None and len(layout_node):
        layout = layout_node[0].tag

    run = root.find(".//RUN")
    run_accession = run.attrib.get("accession", accession) if run is not None else accession

    return {
        "sample_id": sample,
        "run_accession": run_accession,
        "organism": get_text(root, ".//SAMPLE/SAMPLE_NAME/SCIENTIFIC_NAME"),
        "strategy": get_text(root, ".//LIBRARY_DESCRIPTOR/LIBRARY_STRATEGY"),
        "source": get_text(root, ".//LIBRARY_DESCRIPTOR/LIBRARY_SOURCE"),
        "selection": get_text(root, ".//LIBRARY_DESCRIPTOR/LIBRARY_SELECTION"),
        "layout": layout,
        "instrument": get_text(root, ".//INSTRUMENT_MODEL"),
        "experiment_title": get_text(root, ".//EXPERIMENT/TITLE"),
        "experiment_description": get_text(root, ".//DESIGN_DESCRIPTION"),
    }

# Run after XML retrieval:
#
# metadata = pd.DataFrame([
#     parse_sra_xml(row.sample_id, row.run_accession)
#     for row in samples.itertuples()
# ])
#
# metadata.to_csv(BASE / "metadata" / "sra_metadata.tsv", sep="\t", index=False)
# metadata



The agent should read `metadata/sra_metadata.tsv` and create the review table itself.

In [ ]:
import re

def create_metadata_review(metadata):
    reviews = []

    for _, r in metadata.fillna("").iterrows():
        combined = " ".join([
            r["strategy"],
            r["source"],
            r["selection"],
            r["experiment_title"],
            r["experiment_description"],
        ])

        organism_ok = "escherichia coli" in r["organism"].lower()

        if (
            r["strategy"].upper() == "WGS"
            and r["source"].upper() == "GENOMIC"
            and organism_ok
        ):
            status = "VALID_WGS"
            evidence = (
                f"Organism={r['organism']}; Strategy={r['strategy']}; "
                f"Source={r['source']}; Layout={r['layout']}"
            )
        elif re.search(r"16S|V3[-–]?V4|V4|amplicon", combined, re.IGNORECASE):
            status = "MISMATCH"
            evidence = "Metadata indicates targeted 16S/amplicon sequencing rather than WGS."
        else:
            status = "REVIEW"
            evidence = "Metadata is insufficient or conflicting for automatic WGS validation."

        reviews.append({
            "sample_id": r["sample_id"],
            "run_accession": r["run_accession"],
            "wgs_compatible": status == "VALID_WGS",
            "evidence_summary": evidence,
            "status": status,
        })

    review = pd.DataFrame(reviews)
    review.to_csv(BASE / "metadata" / "metadata_review.tsv", sep="\t", index=False)
    return review

# Example:
# metadata = pd.read_csv(BASE / "metadata" / "sra_metadata.tsv", sep="\t")
# review = create_metadata_review(metadata)
# review

# Step 2 — Locate the pre-downloaded WGS reads

Once the appropriate datasets have been identified, we can download their raw sequencing reads using the SRA Toolkit. The agent will retrieve only validated WGS samples and organize their paired-end FASTQ files into the project directory. Original sequencing files will remain unchanged to ensure reproducibility and allow reprocessing if needed.

**Goal:** Identify the samples that passed metadata review, then locate their provided paired-end FASTQ files. **There is no SRA download during the workshop.** The raw reads should stay unchanged.

### Ask your agent

> Use `metadata/metadata_review.tsv` to create `metadata/valid_wgs.tsv` containing only `VALID_WGS` samples. Locate each sample's staged R1 and R2 FASTQ files in the project `raw/` directory, check that files exist, and report filenames and sizes. If any input is missing, stop and tell me what is missing; do not download data, create fake reads, or change the shared inputs. Save a brief file inventory in `reports/input_inventory.tsv`.

**Explore / Checkpoint:** Why must an R1/R2 pair refer to the same sample? Are any non-WGS reads present? **Deliverable:** `metadata/valid_wgs.tsv` and file inventory.

*Optional instructor reference:* the original SRA download example remains below but must stay disabled.

In [ ]:
def write_valid_wgs_table():
    review = pd.read_csv(BASE / "metadata" / "metadata_review.tsv", sep="\t")

    valid = review.loc[
        review["status"] == "VALID_WGS",
        ["sample_id", "run_accession"],
    ]

    valid.to_csv(BASE / "metadata" / "valid_wgs.tsv", sep="\t", index=False)
    return valid

# valid = write_valid_wgs_table()
# valid

In [ ]:
RUN_DOWNLOADS = False  # Instructor-only example; keep False for workshop.

if RUN_DOWNLOADS:
    valid = pd.read_csv(BASE / "metadata" / "valid_wgs.tsv", sep="\t")

    for _, row in valid.iterrows():
        sample = row["sample_id"]
        accession = row["run_accession"]
        outdir = BASE / "raw" / sample

        commands = [
            f"prefetch --max-size u {accession}",
            f"vdb-validate {accession}",
            f"fasterq-dump {accession} --split-files -e 4 -O {outdir}",
            f"pigz -p 4 {outdir}/*.fastq",
        ]

        for cmd in commands:
            print(cmd)
            subprocess.run(cmd, shell=True, check=True)
            with open(BASE / "logs" / "commands.log", "a") as log:
                log.write(cmd + "\n")
else:
    print("Live downloading disabled; use instructor-staged FASTQs.")

# Step 3 — Validate the raw FASTQ files and run initial QC

Raw sequencing reads may contain low-quality bases, adapter contamination, or incomplete files that can interfere with downstream analyses. We will use SeqKit, FastQC, and MultiQC to examine read counts, sequence quality, and other important metrics. The agent will also verify that paired-end files are present and readable, identifying potential problems before preprocessing begins.

**Goal:** Check that the staged FASTQs are readable and inspect their quality before cleaning. We will use SeqKit for sequence statistics, FastQC for detailed plots, and MultiQC for a combined report.

### Ask your agent

> For each sample in `metadata/valid_wgs.tsv`, check that both R1 and R2 are valid, non-empty compressed FASTQ files. Run `seqkit stats`, FastQC, and MultiQC on the raw reads, placing outputs in `qc/raw/`. Use up to four threads per job. Record commands in `logs/commands.log`. Explain the read count and two noteworthy observations from the QC output. Don't install packages or rerun downloads.

**Explore / Checkpoint:** Open the MultiQC HTML report. Find the read lengths and per-base quality plot. Ask the agent to explain one unusual feature. **Deliverable:** raw QC outputs and a short interpretation.

*Reference below:* a Python FASTQ validation helper and example shell commands. The commands are printed, not automatically run.

In [ ]:
import gzip

def validate_fastq_gz(path):
    path = Path(path)

    if not path.exists():
        return False, "missing"
    if path.stat().st_size == 0:
        return False, "empty"

    try:
        with gzip.open(path, "rt") as fh:
            header = fh.readline().strip()
            seq = fh.readline().strip()
            plus = fh.readline().strip()
            qual = fh.readline().strip()

        if not header.startswith("@") or not plus.startswith("+"):
            return False, "FASTQ structure not recognized"
        if len(seq) != len(qual):
            return False, "sequence/quality length mismatch"

    except Exception as exc:
        return False, str(exc)

    return True, "ok"

def validate_all_wgs_fastqs(valid):
    rows = []

    for _, row in valid.iterrows():
        sample = row["sample_id"]
        accession = row["run_accession"]

        r1 = BASE / "raw" / sample / f"{accession}_1.fastq.gz"
        r2 = BASE / "raw" / sample / f"{accession}_2.fastq.gz"

        r1_ok, r1_msg = validate_fastq_gz(r1)
        r2_ok, r2_msg = validate_fastq_gz(r2)

        rows.append({
            "sample_id": sample,
            "run_accession": accession,
            "r1_ok": r1_ok,
            "r1_status": r1_msg,
            "r2_ok": r2_ok,
            "r2_status": r2_msg,
        })

    result = pd.DataFrame(rows)
    result.to_csv(BASE / "reports" / "fastq_validation.tsv", sep="\t", index=False)
    return result

In [ ]:
raw_qc_commands = [
    f"seqkit stats -a -T {BASE}/raw/*/*.fastq.gz > {BASE}/qc/raw/seqkit_raw.tsv",
    f"fastqc -t 4 -o {BASE}/qc/raw/fastqc {BASE}/raw/*/*.fastq.gz",
    f"multiqc {BASE}/qc/raw/fastqc -o {BASE}/qc/raw/multiqc -f",
]

RUN_RAW_QC = False

if RUN_RAW_QC:
    for cmd in raw_qc_commands:
        print(cmd)
        subprocess.run(cmd, shell=True, check=True)
        with open(BASE / "logs" / "commands.log", "a") as log:
            log.write(cmd + "\n")
else:
    print("\n".join(raw_qc_commands))

# Step 4 — Clean the WGS reads using fastp

Preprocessing improves sequencing data quality by removing adapters and filtering low-quality reads. We will use fastp to clean the paired-end FASTQ files while preserving the original data. The agent will construct the preprocessing commands, organize the outputs, and document the parameters used to ensure that the workflow remains reproducible.

**Goal:** Remove sequencing adapters and filter poor-quality reads while preserving raw inputs. Compare decisions with the agent before executing them.

### Ask your agent

> Propose `fastp` commands for the paired-end WGS FASTQs using `--detect_adapter_for_pe` and at most four threads. Show the plan first. After my approval, process each eligible sample into `processed/<sample_id>/`, save JSON/HTML summaries in `reports/fastp/`, and log commands. Do not alter raw FASTQ files or process the metadata mismatch.

**Explore / Checkpoint:** Open one `fastp` HTML report. How many read pairs were kept? What changed? **Deliverable:** cleaned R1/R2 reads and `fastp` summaries.

*Reference below:* example `fastp` command builder; not automatically executed.

In [ ]:
def build_fastp_command(sample, accession):
    r1 = BASE / "raw" / sample / f"{accession}_1.fastq.gz"
    r2 = BASE / "raw" / sample / f"{accession}_2.fastq.gz"
    out1 = BASE / "processed" / sample / f"{sample}_R1.clean.fastq.gz"
    out2 = BASE / "processed" / sample / f"{sample}_R2.clean.fastq.gz"
    html = BASE / "reports" / "fastp" / f"{sample}_fastp.html"
    js = BASE / "reports" / "fastp" / f"{sample}_fastp.json"

    return (
        f"fastp -i {r1} -I {r2} "
        f"-o {out1} -O {out2} "
        f"--detect_adapter_for_pe --thread 4 "
        f"--html {html} --json {js}"
    )

RUN_FASTP = False

if RUN_FASTP:
    valid = pd.read_csv(BASE / "metadata" / "valid_wgs.tsv", sep="\t")

    for _, row in valid.iterrows():
        cmd = build_fastp_command(row["sample_id"], row["run_accession"])
        print(cmd)

        subprocess.run(cmd, shell=True, check=True)

        with open(BASE / "logs" / "commands.log", "a") as log:
            log.write(cmd + "\n")
else:
    print("fastp scaffold ready.")

# Step 5 — Compare before and after quality control

After preprocessing, we need to determine whether read quality improved without losing an excessive amount of sequencing data. We will repeat the QC analysis and compare the processed reads against the original files. The agent will help summarize changes in read retention, sequence quality, and other metrics to identify samples requiring further investigation.

**Goal:** Assess whether cleaning improved quality and how much data was retained. A high quality score does not help if most sequencing data were lost.

### Ask your agent

> Run SeqKit, FastQC, and MultiQC on the processed reads and save them in `qc/processed/`. For each sample compare raw versus processed read counts and key quality metrics. Make a simple bar chart of raw versus processed read-pair counts, save it as `reports/read_counts_before_after.png`, and explain the result in plain language. Do not automatically reject any sample.

**Explore / Checkpoint:** Which sample retained the smallest fraction of reads? Ask the agent what might explain it and which additional QC evidence would help. **Deliverable:** post-QC reports and one visualization.

*Reference below:* QC command examples; the next viewer cell can plot the resulting feature table.

In [ ]:
processed_qc_commands = [
    f"seqkit stats -a -T {BASE}/processed/*/*.fastq.gz > {BASE}/qc/processed/seqkit_processed.tsv",
    f"fastqc -t 4 -o {BASE}/qc/processed/fastqc {BASE}/processed/*/*.fastq.gz",
    f"multiqc {BASE}/qc/processed/fastqc -o {BASE}/qc/processed/multiqc -f",
]

RUN_PROCESSED_QC = False

if RUN_PROCESSED_QC:
    for cmd in processed_qc_commands:
        print(cmd)
        subprocess.run(cmd, shell=True, check=True)
        with open(BASE / "logs" / "commands.log", "a") as log:
            log.write(cmd + "\n")
else:
    print("\n".join(processed_qc_commands))

### Checkpoint — Interpret the changes

Compare the raw and processed reports. Did the quality profiles improve? Were there substantial losses of data? Ask your agent to support every conclusion using the actual metrics rather than just saying the reads look “good.”

# Step 6 — Build and visualize QC features

Sequencing QC generates several metrics that can be combined to evaluate overall sample quality. In this step, we will extract measurements such as read retention, mean Phred score, and sequence length into a structured feature table. This table will serve as input for a machine-learning classifier designed to distinguish acceptable samples from those needing further review.

**Goal:** Turn QC results into a small table, with one row per WGS sample. These measurements will be used in the ML demonstration.

### Ask your agent

> Generate `reports/processed_qc_features.tsv` with `sample_id`, `run_accession`, `retention_percent`, `mean_phred`, `q30_percent`, `n_percent`, and `pair_balance_percent`. Verify that values are numeric and plausible and tell me how each value was calculated. Make one scatter plot of retention versus Q30 and label the samples. Keep computations fast and avoid scanning more reads than needed for sampled quality summaries.

**Explore / Checkpoint:** Are high retention and high Q30 necessarily the same thing? **Deliverable:** valid feature table plus one simple plot.

*Reference below:* the example feature builder estimates quality metrics from up to 25,000 reads per FASTQ; note that this is a **subsampled estimate**, not a full-data measurement.

In [ ]:
import numpy as np

def count_reads(path):
    lines = 0
    with gzip.open(path, "rt") as fh:
        for lines, _ in enumerate(fh, start=1):
            pass
    return lines // 4

def scan_quality(path, max_reads=25000):
    reads = bases = q_sum = q30 = n_count = 0

    with gzip.open(path, "rt") as fh:
        while reads < max_reads:
            header = fh.readline()
            if not header:
                break

            seq = fh.readline().strip()
            fh.readline()
            qual = fh.readline().strip()

            if len(seq) != len(qual):
                raise ValueError(f"Sequence/quality mismatch in {path}")

            q = np.fromiter((ord(c) - 33 for c in qual), dtype=np.int16)

            reads += 1
            bases += len(seq)
            q_sum += int(q.sum())
            q30 += int((q >= 30).sum())
            n_count += seq.upper().count("N")

    if bases == 0:
        raise ValueError(f"No reads found in {path}")

    return {"bases": bases, "q_sum": q_sum, "q30": q30, "n_count": n_count}

def build_qc_feature_table(valid):
    rows = []

    for _, r in valid.iterrows():
        sample = r["sample_id"]
        accession = r["run_accession"]

        raw1 = BASE / "raw" / sample / f"{accession}_1.fastq.gz"
        raw2 = BASE / "raw" / sample / f"{accession}_2.fastq.gz"
        clean1 = BASE / "processed" / sample / f"{sample}_R1.clean.fastq.gz"
        clean2 = BASE / "processed" / sample / f"{sample}_R2.clean.fastq.gz"

        raw_r1, raw_r2 = count_reads(raw1), count_reads(raw2)
        clean_r1, clean_r2 = count_reads(clean1), count_reads(clean2)

        raw_pairs = min(raw_r1, raw_r2)
        clean_pairs = min(clean_r1, clean_r2)

        retention = 100 * clean_pairs / raw_pairs
        pair_balance = 100 * min(clean_r1, clean_r2) / max(clean_r1, clean_r2)

        q1, q2 = scan_quality(clean1), scan_quality(clean2)
        bases = q1["bases"] + q2["bases"]

        rows.append({
            "sample_id": sample,
            "run_accession": accession,
            "retention_percent": retention,
            "mean_phred": (q1["q_sum"] + q2["q_sum"]) / bases,
            "q30_percent": 100 * (q1["q30"] + q2["q30"]) / bases,
            "n_percent": 100 * (q1["n_count"] + q2["n_count"]) / bases,
            "pair_balance_percent": pair_balance,
        })

    features = pd.DataFrame(rows)
    features.to_csv(BASE / "reports" / "processed_qc_features.tsv", sep="\t", index=False)
    return features

# valid = pd.read_csv(BASE / "metadata" / "valid_wgs.tsv", sep="\t")
# features = build_qc_feature_table(valid)
# features

### Explore the feature table

After your agent finishes Step 6, run the viewer below. The scatter plot lets you compare two quality metrics without requiring Python experience.

In [ ]:
features_file = BASE / "reports" / "processed_qc_features.tsv"
if features_file.is_file():
    t = pd.read_csv(features_file, sep="\t")
    display(t)
    if {"retention_percent", "q30_percent", "sample_id"}.issubset(t.columns) and not t.empty:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.scatter(t["retention_percent"], t["q30_percent"])
        for row in t.itertuples():
            ax.annotate(row.sample_id, (row.retention_percent, row.q30_percent), xytext=(4, 4), textcoords="offset points")
        ax.set_xlabel("Read-pair retention (%)"); ax.set_ylabel("Q30 bases (%)")
        ax.set_title("Explore QC metrics"); plt.tight_layout(); plt.show()
else:
    print("No feature table yet. Complete Step 6 first.")

### Exploration — visualize read retention

Use this viewer after Step 6 (the feature table summarizes the processed reads). The agent should also create the raw-versus-processed count plot in Step 5.

In [ ]:
import matplotlib.pyplot as plt
feature_file = BASE / "reports" / "processed_qc_features.tsv"
if feature_file.is_file():
    t = pd.read_csv(feature_file, sep="\t")
    if {"sample_id", "retention_percent"}.issubset(t.columns) and not t.empty:
        ax = t.plot.bar(x="sample_id", y="retention_percent", legend=False, figsize=(6, 3), rot=0)
        ax.set_ylabel("Paired-read retention (%)")
        ax.set_title("Reads retained after preprocessing")
        plt.tight_layout(); plt.show()
else:
    print("Complete Step 6 first to visualize retention percentages.")

# Step 7 — Try a small, constrained machine-learning classifier

We will demonstrate how machine learning can support automated sequencing quality assessment. Using synthetic training data, the agent will compare logistic regression and random forest models and apply the selected classifier to our processed samples. Predictions will classify samples as PASS or REVIEW, providing an additional decision-support tool rather than replacing manual QC assessment.

**Goal:** Demonstrate a simple model predicting `PASS` or `REVIEW` from QC measurements. **The 80 training examples below are synthetic** and are not clinical standards. The model is a teaching example, not an independent validation of sequencing quality.

### Ask your agent

> Use only `reports/processed_qc_features.tsv`, the synthetic training data, and these five columns: `retention_percent`, `mean_phred`, `q30_percent`, `n_percent`, `pair_balance_percent`. Compare scikit-learn logistic regression and a random forest with no more than 100 trees, `random_state=42`, and 3-fold stratified cross-validation. Do not download data, add models, tune hyperparameters, or use deep learning. Report cross-validation macro-F1, fit the chosen model, save `reports/ml_qc_results.tsv`, and make **one** small classification plot. Limit execution to a few seconds for the tiny training dataset. If the feature table is missing, stop and ask me to complete Step 6.

**Explore / Checkpoint:** Does a `PASS` prediction mean the sample is scientifically valid? What could go wrong if synthetic training labels are unrealistic? **Deliverable:** model comparison, predictions, and one plot.

*Reference below:* small synthetic training data and bounded example classifiers. The prediction cell checks whether Step 6 produced the necessary input.


### Synthetic demonstration data (provided code)

Because three (maybe more) WGS samples are insufficient for supervised training, this cell creates a fixed, small training dataset for demonstration. You may run it as provided. It is **not** learned from the real workshop samples, and its labels should not be treated as ground truth.

In [ ]:
rng = np.random.default_rng(42)

n_pass = 50
n_review = 30

pass_df = pd.DataFrame({
    "retention_percent": rng.uniform(88, 100, n_pass),
    "mean_phred": rng.uniform(32, 38, n_pass),
    "q30_percent": rng.uniform(82, 98, n_pass),
    "n_percent": rng.uniform(0.0, 0.30, n_pass),
    "pair_balance_percent": rng.uniform(99.5, 100, n_pass),
    "label": "PASS",
})

review_df = pd.DataFrame({
    "retention_percent": rng.uniform(35, 90, n_review),
    "mean_phred": rng.uniform(22, 32, n_review),
    "q30_percent": rng.uniform(40, 84, n_review),
    "n_percent": rng.uniform(0.2, 4.0, n_review),
    "pair_balance_percent": rng.uniform(92, 100, n_review),
    "label": "REVIEW",
})

training = pd.concat([pass_df, review_df], ignore_index=True)
training.sample(8, random_state=1)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

FEATURES = [
    "retention_percent",
    "mean_phred",
    "q30_percent",
    "n_percent",
    "pair_balance_percent",
]

X = training[FEATURES]
y = training["label"]

models = {
    "logistic_regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(random_state=42)),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=["accuracy", "precision_macro", "recall_macro", "f1_macro"],
    )

    rows.append({
        "model": name,
        "accuracy": scores["test_accuracy"].mean(),
        "precision_macro": scores["test_precision_macro"].mean(),
        "recall_macro": scores["test_recall_macro"].mean(),
        "f1_macro": scores["test_f1_macro"].mean(),
    })

comparison = pd.DataFrame(rows).sort_values("f1_macro", ascending=False)
comparison.to_csv(BASE / "reports" / "model_comparison.tsv", sep="\t", index=False)
comparison

In [ ]:
best_name = comparison.iloc[0]["model"]
best_model = models[best_name]
best_model.fit(X, y)

qc = pd.read_csv(BASE / "reports" / "processed_qc_features.tsv", sep="\t")

qc["ml_qc_class"] = best_model.predict(qc[FEATURES])

classes = list(best_model.classes_)
pass_index = classes.index("PASS")
qc["pass_probability"] = best_model.predict_proba(qc[FEATURES])[:, pass_index]

qc.to_csv(BASE / "reports" / "ml_qc_results.tsv", sep="\t", index=False)
qc

### Explore classification results

The synthetic classifier is intentionally small. Ask your agent to explain the plotted classes in terms of the underlying QC metrics, not to interpret the probabilities as validated risk estimates.

In [ ]:
pred_file = BASE / "reports" / "ml_qc_results.tsv"
if pred_file.is_file():
    t = pd.read_csv(pred_file, sep="\t")
    if {"retention_percent", "q30_percent", "ml_qc_class"}.issubset(t.columns) and not t.empty:
        fig, ax = plt.subplots(figsize=(6, 4))
        for lab, g in t.groupby("ml_qc_class"):
            ax.scatter(g["retention_percent"], g["q30_percent"], label=lab)
        ax.set_xlabel("Retention (%)"); ax.set_ylabel("Q30 (%)")
        ax.legend(title="Illustrative class"); plt.tight_layout(); plt.show()
else:
    print("No model predictions yet; complete Step 7.")

# Step 8 — Interpret findings and create a report

The final step consolidates the workflow results into a reproducible report. The agent will summarize metadata validation, preprocessing outcomes, QC metrics, and machine-learning classifications for each submitted sample. This report provides a record of what was performed, which samples passed evaluation, and which may require further attention.

**Goal:** Turn the workflow's results into an understandable, reproducible summary. Your scientific judgment matters: the agent can summarize evidence, but it cannot decide whether a sample is appropriate for every downstream experiment.

### Ask your agent

> Create `reports/workflow_report.md` explaining which accessions matched WGS, what raw data were available, what preprocessing and QC showed, and which samples need follow-up. Include the plotted QC comparisons, model predictions *with the caveat that training labels were synthetic*, output locations, software versions, and logged commands. Use clear language for non-bioinformaticians. Do not claim that the model certifies sample quality. If files are missing, state that rather than inventing results.

**Explore / Checkpoint:** Ask your agent to identify one uncertain conclusion and the next piece of evidence needed to resolve it. Compare the report with the actual QC plots and tables. **Deliverable:** `reports/workflow_report.md` and `reports/final_summary.tsv`.

In [ ]:
review_path = BASE / "metadata" / "metadata_review.tsv"
ml_path = BASE / "reports" / "ml_qc_results.tsv"
if review_path.is_file() and ml_path.is_file():
    metadata_review = pd.read_csv(review_path, sep="\t")
    ml_results = pd.read_csv(ml_path, sep="\t")
    final_summary = metadata_review.merge(ml_results, on=["sample_id", "run_accession"], how="left")
    final_summary.to_csv(BASE / "reports" / "final_summary.tsv", sep="\t", index=False)
    display(final_summary)
else:
    print("Summary needs metadata_review.tsv and ml_qc_results.tsv; complete earlier steps first.")

## Expected output checklist

- `metadata/metadata_review.tsv` and `metadata/valid_wgs.tsv`
- `reports/input_inventory.tsv` and checked raw FASTQ inputs
- `qc/raw/` and `qc/processed/` with SeqKit/FastQC/MultiQC reports
- `processed/<sample_id>/` paired cleaned FASTQ files; `reports/fastp/` summaries
- `reports/read_counts_before_after.png` and at least one additional exploratory plot
- `reports/processed_qc_features.tsv`, `reports/model_comparison.tsv`, `reports/ml_qc_results.tsv`
- `reports/workflow_report.md`, `reports/final_summary.tsv`, and `logs/commands.log`

**If you finish early:** Ask the agent to create a different QC visualization or a concise plain-language interpretation, then verify whether the plot actually supports its explanation. Do not launch extra large jobs.


## Future downstream analyses

This workshop (for now) stops after WGS validation, preprocessing, post-processing QC, and ML-based quality assessment, but the processed reads could serve as the starting point for a more complete bacterial genomics workflow. Possible downstream analyses:

1.   de novo assembly (SPAdes)
2.   genome annotation (Bakta)
3.   antimicrobial resistance detection with AMRFinderPlus
4.   Other options people may think of



# Instructor answer key

**Remove this cell before distributing the final participant version.**

`ERR4567643` → valid *E. coli* WGS  
`SRR25073407` → valid *E. coli* WGS  
`SRR29154704` → valid *E. coli* WGS  
`SRR27836258` → deliberate 16S / non-WGS mismatch; flag before download and preprocessing